# Imports & DB Path

In [30]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

import scipy.stats as stats
from scipy.stats import binomtest
from scipy.stats import chi2_contingency

import matplotlib.pyplot as plt
import numpy as np

# __file__ isn't defined in notebooks, so resolve relative to the current working
# directory. Search upward for the "backend" dir so this works regardless of where
# the kernel was started.
_here = Path.cwd().resolve()
_backend = next(p for p in (_here, *_here.parents) if (p / "database" / "coup_generated.sqlite3").exists())
DB_PATH = _backend / "database" / "coup_generated.sqlite3"
MODELS = ["gemini-3.1-flash-lite", "gpt-5.4-nano", "claude-haiku-4-5"]
CARDS = ["Duke", "Assassin", "Captain", "Ambassador", "Contessa"]

# PD Dataframes

In [6]:
conn = sqlite3.connect(DB_PATH)

# 2. Extract the tables into DataFrames.
states = pd.read_sql_query("SELECT * FROM State", conn)
playersnapshots = pd.read_sql_query("SELECT * FROM PlayerSnapshot", conn)
games = pd.read_sql_query("SELECT * FROM Game", conn)
decisions = pd.read_sql_query("SELECT * FROM Decision", conn)
results = pd.read_sql_query("SELECT * FROM Result", conn)

conn.close()

print(f"games: {len(games)} rows")
print(games.columns.tolist())
print(f"states: {len(states)} rows")
print(states.columns.tolist())
print(f"decisions: {len(decisions)} rows")
print(decisions.columns.tolist())
print(f"results: {len(results)} rows")
print(results.columns.tolist())
print(f"playersnapshots: {len(playersnapshots)} rows")
print(playersnapshots.columns.tolist())

observed = results.groupby("winner_name").size()

games: 126 rows
['game_id']
states: 5861 rows
['state_id', 'game_id', 'experiment_id', 'state_seq', 'turn_id', 'phase', 'player_ids', 'acting_player_id', 'deck', 'discard_pile', 'victim_id', 'blocked', 'challenged', 'blocker_id', 'challenger_id']
decisions: 5724 rows
['decision_id', 'game_id', 'state_id', 'player_id', 'provider', 'model', 'decision_type', 'action', 'claimed_card', 'is_bluff', 'target_player_id', 'resolved_successfully', 'legal_actions', 'raw_decision']
results: 125 rows
['game_id', 'winner_name', 'winner_id', 'total_turns', 'total_states']
playersnapshots: 17581 rows
['state_id', 'player_id', 'player_name', 'cards', 'num_coins', 'active']


# Winrates Tests

In [7]:
print("--WINRATES--")
winrates = observed / observed.sum()
winrates = winrates.sort_values(ascending=False)
print(winrates)
print()

print("Chi2 test:")
chi2_stat, p_value = stats.chisquare(f_obs=observed)
print(f"Chi2 stat: {chi2_stat}")
print(f"P-value: {p_value}")
print()

print("Binomial test:")
binom_test = binomtest(observed.sort_values(ascending=False).iloc[0], n=observed.sum(), p=1/3)
print(f"P-value: {binom_test.pvalue}")

--WINRATES--
winner_name
gemini-3.1-flash-lite    0.516129
gpt-5.4-nano             0.314516
claude-haiku-4-5         0.169355
dtype: float64

Chi2 test:
Chi2 stat: 22.564516129032256
P-value: 1.2594402805149302e-05

Binomial test:
P-value: 3.382064318598434e-05


# Card Configuration Tests

In [ ]:
print("--STARTING HANDS--")
# print()

starting_hands = {m: [] for m in MODELS}

# player_name in PlayerSnapshot is only "Player 1/2/3", and player_id (1/2/3)
# is reused across games, so the model for a seat is game-specific. Build a
# (game_id, player_id) -> model mapping from the Decision table.
player_models = decisions.groupby(["game_id", "player_id"])["model"].first()

# The starting state of each game is the one with the lowest state_seq.
first_states = (
    states.sort_values("state_seq")
    .groupby("game_id")
    .first()
    .reset_index()[["game_id", "state_id"]]
)

for game_id, state_id in first_states.itertuples(index=False):
    snap = playersnapshots[playersnapshots["state_id"] == state_id]
    for _, row in snap.iterrows():
        model = player_models.get((game_id, row["player_id"]))
        if model in starting_hands:
            # cards is stored as a JSON string, e.g. '["Contessa", "Captain"]'.
            starting_hands[model].append(json.loads(row["cards"]))

# print(len(starting_hands["gpt-5.4-nano"]))

all_configs = set()
for idx, c1 in enumerate(CARDS):
    for c2 in CARDS[idx:]:
        all_configs.add(tuple(sorted((c1, c2))))

assert len(all_configs) == 15, "There should be 15 unique card configurations"
expected = {cfg: len(starting_hands[MODELS[0]]) / len(all_configs) for cfg in all_configs}

starting_hands_counts = {m: {cfg: 0 for cfg in all_configs} for m in MODELS}

for model in MODELS:
    for hand in starting_hands[model]:
        starting_hands_counts[model][tuple(sorted(hand))] += 1

print("Test for each model:")
print()
for m in MODELS:
    print(f"{m}:")
    observed = starting_hands_counts[m]
    print(f"Average count: {np.mean(list(observed.values()))}")
    print(f"Expected count: {np.mean(list(expected.values()))}")
    chi2_stat, p_value = stats.chisquare(f_obs=list(observed.values()), f_exp=list(expected.values()))
    print(f"Chi2 stat: {chi2_stat}")
    print(f"P-value: {p_value}")
    print()

--STARTING HANDS--
Test for each model:

gemini-3.1-flash-lite:
Average count: 8.333333333333334
Expected counts: 8.333333333333332
Chi2 stat: 30.399999999999995
P-value: 0.006719980567492036

gpt-5.4-nano:
Average count: 8.333333333333334
Expected counts: 8.333333333333332
Chi2 stat: 37.36
P-value: 0.0006505811922971715

claude-haiku-4-5:
Average count: 8.333333333333334
Expected counts: 8.333333333333332
Chi2 stat: 46.0
P-value: 2.8037299014042824e-05



In [32]:
homogeneity_data = pd.DataFrame(starting_hands_counts)

chi2_stat, p_val, dof, expected_frequencies = chi2_contingency(homogeneity_data)

# Print results
print("--HOMOGENEITY TEST--")
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Degrees of Freedom: {dof}")
print("\nExpected Frequencies Table:\n", expected_frequencies)

--HOMOGENEITY TEST--
Chi-Square Statistic: 32.6532
p-value: 0.2488
Degrees of Freedom: 28

Expected Frequencies Table:
 [[ 4.          4.          4.        ]
 [ 9.33333333  9.33333333  9.33333333]
 [10.66666667 10.66666667 10.66666667]
 [ 9.33333333  9.33333333  9.33333333]
 [11.33333333 11.33333333 11.33333333]
 [ 2.33333333  2.33333333  2.33333333]
 [ 1.33333333  1.33333333  1.33333333]
 [10.66666667 10.66666667 10.66666667]
 [ 9.66666667  9.66666667  9.66666667]
 [12.33333333 12.33333333 12.33333333]
 [13.         13.         13.        ]
 [ 3.66666667  3.66666667  3.66666667]
 [ 5.33333333  5.33333333  5.33333333]
 [14.         14.         14.        ]
 [ 8.          8.          8.        ]]


# GEMINI

In [42]:
GEMINI = "gemini-3.1-flash-lite"

# Key gemini's starting hand by game_id so it can be aligned with results.
gemini_hand_by_game = {}
for game_id, state_id in first_states.itertuples(index=False):
    snap = playersnapshots[playersnapshots["state_id"] == state_id]
    for _, row in snap.iterrows():
        if player_models.get((game_id, row["player_id"])) == GEMINI:
            hand = sorted(json.loads(row["cards"]))
            gemini_hand_by_game[game_id] = f"{hand[0]}-{hand[1]}"

results_for_gemini = {
    game_id: "Win" if results[results["game_id"] == game_id]["winner_name"].iloc[0] == GEMINI else "Loss"
    for game_id in results["game_id"].unique()
}

# Align on games that appear in both.
common_games = [g for g in results_for_gemini if g in gemini_hand_by_game]
gemini_data = pd.DataFrame(data={
    "Starting Hand": [gemini_hand_by_game[g] for g in common_games],
    "Result": [results_for_gemini[g] for g in common_games],
})

print("--STARTING HANDS VS RESULT--")
gemini_contingency = pd.crosstab(gemini_data["Starting Hand"], gemini_data["Result"])
print(gemini_contingency)

chi2_stat, p_val, dof, expected_frequencies = chi2_contingency(gemini_contingency)
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Degrees of Freedom: {dof}")

--STARTING HANDS VS RESULT--
Result                 Loss  Win
Starting Hand                   
Ambassador-Ambassador     0    3
Ambassador-Assassin      10    3
Ambassador-Captain        2    6
Ambassador-Contessa      12    1
Ambassador-Duke           3    7
Assassin-Captain          7    4
Assassin-Contessa         7    3
Assassin-Duke             2    8
Captain-Captain           0    1
Captain-Contessa          7    4
Captain-Duke              2    8
Contessa-Contessa         2    4
Contessa-Duke             4    9
Duke-Duke                 3    3
Chi-Square Statistic: 33.6504
p-value: 0.0014
Degrees of Freedom: 13


# CLAUDE

In [43]:
CLAUDE = "claude-haiku-4-5"

# Key gemini's starting hand by game_id so it can be aligned with results.
claude_hand_by_game = {}
for game_id, state_id in first_states.itertuples(index=False):
    snap = playersnapshots[playersnapshots["state_id"] == state_id]
    for _, row in snap.iterrows():
        if player_models.get((game_id, row["player_id"])) == GEMINI:
            hand = sorted(json.loads(row["cards"]))
            claude_hand_by_game[game_id] = f"{hand[0]}-{hand[1]}"

results_for_claude = {
    game_id: "Win" if results[results["game_id"] == game_id]["winner_name"].iloc[0] == CLAUDE else "Loss"
    for game_id in results["game_id"].unique()
}

# Align on games that appear in both.
common_games = [g for g in results_for_claude if g in claude_hand_by_game]
claude_data = pd.DataFrame(data={
    "Starting Hand": [claude_hand_by_game[g] for g in common_games],
    "Result": [results_for_claude[g] for g in common_games],
})

print("--STARTING HANDS VS RESULT--")
claude_contingency = pd.crosstab(claude_data["Starting Hand"], claude_data["Result"])
print(claude_contingency)

chi2_stat, p_val, dof, expected_frequencies = chi2_contingency(claude_contingency)
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Degrees of Freedom: {dof}")

--STARTING HANDS VS RESULT--
Result                 Loss  Win
Starting Hand                   
Ambassador-Ambassador     3    0
Ambassador-Assassin       9    4
Ambassador-Captain        8    0
Ambassador-Contessa      12    1
Ambassador-Duke           8    2
Assassin-Captain          9    2
Assassin-Contessa         6    4
Assassin-Duke             9    1
Captain-Captain           1    0
Captain-Contessa          9    2
Captain-Duke             10    0
Contessa-Contessa         4    2
Contessa-Duke            10    3
Duke-Duke                 6    0
Chi-Square Statistic: 14.0649
p-value: 0.3693
Degrees of Freedom: 13
